# E13_MLP_SignedLog2_Processing

Notebook tiền xử lý riêng cho **E13 – MLP Signed-log + StandardScaler, 128-64**. Notebook đọc Train/Validation đã chia sẵn, xử lý dữ liệu theo cấu hình thực nghiệm, kiểm tra đầu ra và lưu dữ liệu để notebook E tương ứng sử dụng.


# 1. Khai báo đường dẫn và cấu hình


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display

TARGET = "Attack_type"
CATEGORICAL_FEATURES = ["proto", "service"]
EXCLUDED_COLUMNS = [TARGET, "Unnamed: 0", "group_id"]

CURRENT = Path.cwd().resolve()
ROOT = next(
    (
        parent
        for parent in [CURRENT, *CURRENT.parents]
        if (parent / "project").is_dir() and (parent / "result").is_dir()
    ),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Không tìm thấy thư mục gốc sample_project")

SOURCE_DIR = ROOT / "splitted data"
TRAIN_SOURCE = SOURCE_DIR / "train.csv"
VALIDATION_SOURCE = SOURCE_DIR / "validation.csv"
RESULT_DIR = (
    ROOT
    / "result"
    / "preprocessing"
    / "E13_MLP_SignedLog2_Processing"
)
DATA_DIR = RESULT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Thực nghiệm:", "E13")
print("Nguồn Train:", TRAIN_SOURCE)
print("Nguồn Validation:", VALIDATION_SOURCE)
print("Thư mục dữ liệu sau tiền xử lý:", DATA_DIR)


Thực nghiệm: E13
Nguồn Train: D:\SGU26K2_MayHoc_DCT124C3_Nhom4_RT-IoT2022\final_IoT\sample_project\splitted data\train.csv.gz
Nguồn Validation: D:\SGU26K2_MayHoc_DCT124C3_Nhom4_RT-IoT2022\final_IoT\sample_project\splitted data\validation.csv.gz
Thư mục dữ liệu sau tiền xử lý: D:\SGU26K2_MayHoc_DCT124C3_Nhom4_RT-IoT2022\final_IoT\sample_project\result\preprocessing\E13_MLP_SignedLog2_Processing\data


# 2. Đọc Train và Validation đã chia sẵn


In [2]:
for data_path in (TRAIN_SOURCE, VALIDATION_SOURCE):
    if not data_path.is_file():
        raise FileNotFoundError(data_path)

train_df = pd.read_csv(TRAIN_SOURCE)
validation_df = pd.read_csv(VALIDATION_SOURCE)

if TARGET not in train_df.columns or TARGET not in validation_df.columns:
    raise ValueError(f"Train và Validation phải có cột nhãn {TARGET!r}")
if not train_df.columns.equals(validation_df.columns):
    raise ValueError("Tên hoặc thứ tự cột Train/Validation không khớp")
if train_df.empty or validation_df.empty:
    raise ValueError("Train và Validation không được để trống")
if train_df[TARGET].isna().any() or validation_df[TARGET].isna().any():
    raise ValueError("Nhãn có giá trị thiếu")

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
display(train_df.head(3))


Train: (82536, 83)
Validation: (17686, 83)


,id.orig_p,id.resp_p,proto,service,flow_duration,fwd_pkts_tot,bwd_pkts_tot,fwd_data_pkts_tot,bwd_data_pkts_tot,fwd_pkts_per_sec,...,active.std,idle.min,idle.max,idle.tot,idle.avg,idle.std,fwd_init_window_size,bwd_init_window_size,fwd_last_window_size,Attack_type
0,64177,46532,udp,-,0.000000,1,0,1,0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,NMAP_UDP_SCAN
1,20209,21,tcp,-,0.000005,1,1,1,0,199728.761905,...,0.0,0.0,0.0,0.0,0.0,0.0,64,0,64,DOS_SYN_Hping
2,3694,21,tcp,-,0.000005,1,1,1,0,199728.761905,...,0.0,0.0,0.0,0.0,0.0,0.0,64,0,64,DOS_SYN_Hping


# 3. Phân loại đặc trưng và kiểm tra dữ liệu


In [3]:
feature_columns = [
    column for column in train_df.columns if column not in EXCLUDED_COLUMNS
]
missing_categories = [
    column for column in CATEGORICAL_FEATURES if column not in feature_columns
]
if missing_categories:
    raise ValueError(f"Thiếu đặc trưng phân loại: {missing_categories}")

numeric_features = (
    train_df[feature_columns].select_dtypes(include=np.number).columns.tolist()
)
unknown_features = sorted(
    set(feature_columns) - set(numeric_features) - set(CATEGORICAL_FEATURES)
)
if unknown_features:
    raise ValueError(f"Có đặc trưng chưa được phân loại: {unknown_features}")

for frame, label in ((train_df, "Train"), (validation_df, "Validation")):
    if frame[feature_columns].isna().any().any():
        raise ValueError(f"{label} có giá trị thiếu")
    if not np.isfinite(frame[numeric_features].to_numpy(dtype=float)).all():
        raise ValueError(f"{label} có giá trị số NaN/Inf")

labels = sorted(train_df[TARGET].astype(str).unique().tolist())
unknown_labels = sorted(
    set(validation_df[TARGET].astype(str).unique()) - set(labels)
)
if unknown_labels:
    raise ValueError(f"Validation có nhãn không xuất hiện trong Train: {unknown_labels}")

feature_summary = pd.DataFrame(
    [
        {"Nhóm đặc trưng": "Numeric", "Số lượng": len(numeric_features)},
        {
            "Nhóm đặc trưng": "Categorical",
            "Số lượng": len(CATEGORICAL_FEATURES),
        },
        {"Nhóm đặc trưng": "Nhãn", "Số lượng": 1},
    ]
)
print("Số nhãn:", len(labels))
display(feature_summary)
display(train_df[TARGET].value_counts().rename("Số mẫu Train").to_frame())


Số nhãn: 12


,Nhóm đặc trưng,Số lượng
0,Numeric,80
1,Categorical,2
2,Nhãn,1


,Số mẫu Train
Attack_type,
DOS_SYN_Hping,63062
Thing_Speak,5355
ARP_poisioning,5334
MQTT_Publish,2899
NMAP_UDP_SCAN,1807
NMAP_XMAS_TREE_SCAN,1407
NMAP_OS_DETECTION,1400
NMAP_TCP_scan,701
DDOS_Slowloris,373


# 4. Tiền xử lý riêng cho E13

Áp dụng Signed-log cho các đặc trưng số.

Không biến đổi `Attack_type`. `proto` và `service` được giữ nguyên; OneHotEncoder được fit trong Pipeline của thực nghiệm ở từng fold CV.


In [4]:
def signed_log(values):
    values = np.asarray(values, dtype=np.float64)
    return np.sign(values) * np.log1p(np.abs(values))


train_processed = train_df.copy()
validation_processed = validation_df.copy()
train_before = train_df[numeric_features].head(5).copy()
validation_before = validation_df[numeric_features].head(5).copy()

train_processed.loc[:, numeric_features] = signed_log(
    train_processed[numeric_features].to_numpy(dtype=np.float64)
)
validation_processed.loc[:, numeric_features] = signed_log(
    validation_processed[numeric_features].to_numpy(dtype=np.float64)
)

preview = pd.DataFrame(
    {
        "Trước Signed-log": train_before.iloc[:, 0],
        "Sau Signed-log": train_processed[numeric_features].head(5).iloc[:, 0],
    }
)
print("Đặc trưng số được xử lý:", len(numeric_features))
print("Cột phân loại được giữ nguyên:", CATEGORICAL_FEATURES)
display(preview)


Đặc trưng số được xử lý: 80
Cột phân loại được giữ nguyên: ['proto', 'service']


C:\Users\daide\AppData\Local\Temp\ipykernel_35268\2703858034.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[11.06941575  9.91393281  8.21473583 ... 10.53422727 10.58625606
 10.12731087]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train_processed.loc[:, numeric_features] = signed_log(
C:\Users\daide\AppData\Local\Temp\ipykernel_35268\2703858034.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[10.74791702  3.09104245  3.09104245 ...  3.09104245  6.09582456
  3.09104245]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train_processed.loc[:, numeric_features] = signed_log(
C:\Users\daide\AppData\Local\Temp\ipykernel_35268\2703858034.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. V

,Trước Signed-log,Sau Signed-log
0,64177,11.069416
1,20209,9.913933
2,3694,8.214736
3,39875,10.593530
4,9369,9.145268


# 5. Kiểm tra dữ liệu sau tiền xử lý


In [5]:
if not train_processed.columns.equals(train_df.columns):
    raise ValueError("Tiền xử lý đã làm thay đổi cấu trúc cột Train")
if not validation_processed.columns.equals(validation_df.columns):
    raise ValueError("Tiền xử lý đã làm thay đổi cấu trúc cột Validation")
if not train_processed[TARGET].equals(train_df[TARGET]):
    raise ValueError("Không được biến đổi cột nhãn Train")
if not validation_processed[TARGET].equals(validation_df[TARGET]):
    raise ValueError("Không được biến đổi cột nhãn Validation")

for column in CATEGORICAL_FEATURES:
    if not train_processed[column].equals(train_df[column]):
        raise ValueError(f"Không được biến đổi cột phân loại Train: {column}")
    if not validation_processed[column].equals(validation_df[column]):
        raise ValueError(f"Không được biến đổi cột phân loại Validation: {column}")

if not np.isfinite(train_processed[numeric_features].to_numpy(dtype=float)).all():
    raise ValueError("Train sau xử lý có giá trị số NaN/Inf")
if not np.isfinite(
    validation_processed[numeric_features].to_numpy(dtype=float)
).all():
    raise ValueError("Validation sau xử lý có giá trị số NaN/Inf")

if "signed_log" == "none":
    if not np.array_equal(
        train_processed[numeric_features].to_numpy(),
        train_df[numeric_features].to_numpy(),
    ):
        raise ValueError("E11 phải giữ nguyên đặc trưng số")
else:
    expected = signed_log(train_df[numeric_features].to_numpy(dtype=np.float64))
    if not np.allclose(
        train_processed[numeric_features].to_numpy(dtype=np.float64),
        expected,
    ):
        raise ValueError("Kết quả Signed-log không khớp công thức")

check_summary = pd.DataFrame(
    [
        {"Kiểm tra": "Số dòng Train", "Kết quả": len(train_processed)},
        {"Kiểm tra": "Số dòng Validation", "Kết quả": len(validation_processed)},
        {"Kiểm tra": "Số đặc trưng số", "Kết quả": len(numeric_features)},
        {"Kiểm tra": "Số đặc trưng phân loại", "Kết quả": len(CATEGORICAL_FEATURES)},
        {"Kiểm tra": "Số nhãn", "Kết quả": len(labels)},
        {"Kiểm tra": "Giá trị thiếu/NaN/Inf", "Kết quả": "Không phát hiện"},
    ]
)
display(check_summary)


,Kiểm tra,Kết quả
0,Số dòng Train,82536
1,Số dòng Validation,17686
2,Số đặc trưng số,80
3,Số đặc trưng phân loại,2
4,Số nhãn,12
5,Giá trị thiếu/NaN/Inf,Không phát hiện


# 6. Lưu Train/Validation đã xử lý

Dữ liệu được lưu riêng theo từng E trong `result/preprocessing`. Chạy notebook thực nghiệm E tương ứng sau bước này. Scaler (nếu có) vẫn được fit trong từng fold CV để tránh rò rỉ dữ liệu.


In [6]:
TRAIN_OUTPUT = DATA_DIR / "train.csv.gz"
VALIDATION_OUTPUT = DATA_DIR / "validation.csv.gz"
train_processed.to_csv(TRAIN_OUTPUT, index=False, compression="gzip")
validation_processed.to_csv(VALIDATION_OUTPUT, index=False, compression="gzip")

config = {
    "experiment": "E13",
    "preprocessing_notebook": "E13_MLP_SignedLog2_Processing.ipynb",
    "source_train": str(TRAIN_SOURCE),
    "source_validation": str(VALIDATION_SOURCE),
    "processed_train": str(TRAIN_OUTPUT),
    "processed_validation": str(VALIDATION_OUTPUT),
    "numeric_features": numeric_features,
    "numeric_transform": "signed_log",
    "numeric_scaler": "StandardScaler (fit riêng trong mỗi fold CV)",
    "categorical_features": CATEGORICAL_FEATURES,
    "categorical_encoding": "OneHotEncoder in experiment Pipeline, fitted per CV fold",
    "train_samples": int(len(train_processed)),
    "validation_samples": int(len(validation_processed)),
    "test_used": False,
}
with (RESULT_DIR / "E13_processing_config.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(config, file, ensure_ascii=False, indent=2)

print("Đã lưu dữ liệu Train:", TRAIN_OUTPUT)
print("Đã lưu dữ liệu Validation:", VALIDATION_OUTPUT)
display(
    pd.DataFrame(
        [
            {"Tập": "Train", "Số dòng": len(train_processed), "Số cột": len(train_processed.columns)},
            {"Tập": "Validation", "Số dòng": len(validation_processed), "Số cột": len(validation_processed.columns)},
        ]
    )
)


Đã lưu dữ liệu Train: D:\SGU26K2_MayHoc_DCT124C3_Nhom4_RT-IoT2022\final_IoT\sample_project\result\preprocessing\E13_MLP_SignedLog2_Processing\data\train.csv.gz
Đã lưu dữ liệu Validation: D:\SGU26K2_MayHoc_DCT124C3_Nhom4_RT-IoT2022\final_IoT\sample_project\result\preprocessing\E13_MLP_SignedLog2_Processing\data\validation.csv.gz


,Tập,Số dòng,Số cột
0,Train,82536,83
1,Validation,17686,83


### Kết luận

Đã tạo dữ liệu đầu vào riêng cho E13 từ các tập đã chia sẵn. Phương pháp số: **signed_log**. Scaler: **StandardScaler (fit riêng trong mỗi fold CV)**. Test không được sử dụng.
